# TF 2.0 によるロジスティック回帰入門

## 学習目標

1. モデルを構築する
2. サンプルデータでこのモデルを訓練する
3. 未知のデータに対する予測を行うためにモデルを使用する

## はじめに

このラボでは、機械学習を使用してアヤメ（Iris）の花を品種ごとに*分類*します。ここでは TensorFlow を使用して以下を行います。

* TensorFlow のデフォルトである [eager execution（即時実行）](../../guide/eager.ipynb) 開発環境を使用する
* [Datasets API](../../guide/datasets.ipynb) を使用してデータをインポートする
* TensorFlow の [Keras API](../../guide/keras/overview.ipynb) を使用してモデルとレイヤを構築する

このチュートリアルは、多くの TensorFlow プログラムと同様の構造になっています。

1. データセットのインポートと解析
2. モデルタイプの選択
3. モデルの訓練
4. モデルの有効性の評価
5. 訓練済みモデルを使用した予測の実行

## プログラムのセットアップ

### インポートの構成

TensorFlow およびその他の必要な Python モジュールをインポートします。デフォルトでは、TensorFlow は [eager execution（即時実行）](../../guide/eager.ipynb) を使用して操作を即座に評価し、後で実行される計算グラフを作成する代わりに具体的な値を返します。REPL や `python` の対話型コンソールに慣れている方には、おなじみの動作に感じられるでしょう。

In [ ]:
import os

import matplotlib.pyplot as plt

In [ ]:
import tensorflow as tf

In [ ]:
print(f"TensorFlow version: {tf.__version__}")
print(f"Eager execution: {tf.executing_eagerly()}")

## アヤメの分類問題

あなたが見つけた各アヤメの花を自動的に分類する方法を探している植物学者だと想像してみてください。機械学習は、花を統計的に分類するための多くのアルゴリズムを提供します。例えば、高度な機械学習プログラムであれば写真に基づいて花を分類できるかもしれません。ここでの目標はより控えめで、[萼片（がくへん / sepal）](https://en.wikipedia.org/wiki/Sepal) と [花弁（花びら / petal）](https://en.wikipedia.org/wiki/Petal) の長さと幅の測定値に基づいてアヤメの花を分類します。

アヤメ属（Iris）には約 300 の種がありますが、このプログラムでは次の 3 種のみを分類します。

* Iris setosa
* Iris virginica
* Iris versicolor

<table>
  <tr><td>
    <img src="https://www.tensorflow.org/images/iris_three_species.jpg"
         alt="Petal geometry compared for three iris species: Iris setosa, Iris virginica, and Iris versicolor">
  </td></tr>
  <tr><td align="center">
    <b>図 1.</b> <a href="https://commons.wikimedia.org/w/index.php?curid=170298">Iris setosa</a>（作成者: <a href="https://commons.wikimedia.org/wiki/User:Radomil">Radomil</a>, CC BY-SA 3.0）、<a href="https://commons.wikimedia.org/w/index.php?curid=248095">Iris versicolor</a>（作成者: <a href="https://commons.wikimedia.org/wiki/User:Dlanglois">Dlanglois</a>, CC BY-SA 3.0）、および <a href="https://www.flickr.com/photos/33397993@N05/3352169862">Iris virginica</a>（作成者: <a href="https://www.flickr.com/photos/33397993@N05">Frank Mayfield</a>, CC BY-SA 2.0）。<br/>&nbsp;
  </td></tr>
</table>

幸いなことに、萼片と花弁の測定値を含む [120 本のアヤメの花のデータセット](https://en.wikipedia.org/wiki/Iris_flower_data_set) がすでに作成されています。これは、初心者の機械学習の分類問題で人気のある古典的なデータセットです。

## 訓練データセットのインポートと解析

データセットファイルをダウンロードし、この Python プログラムで使用できる構造に変換します。

### データセットのダウンロード

`tf.keras.utils.get_file` 関数を使用して訓練データセットファイルをダウンロードします。これにより、ダウンロードされたファイルのファイルパスが返されます。

In [ ]:
train_dataset_url = "https://storage.googleapis.com/download.tensorflow.org/data/iris_training.csv"

train_dataset_fp = tf.keras.utils.get_file(
    fname=os.path.basename(train_dataset_url), origin=train_dataset_url
)

print(f"Local copy of the dataset file: {train_dataset_fp}")

### データの確認

このデータセット `iris_training.csv` は、カンマ区切り値（CSV）形式の表形式データを格納するプレーンテキストファイルです。`head -n5` コマンドを使用して最初の 5 つのエントリを確認してみましょう。

In [ ]:
!head -n5 {train_dataset_fp}

データセットのこの表示から、以下の点に注目してください。

1. 最初の行は、データセットに関する情報を含むヘッダーです。
  * 合計 120 件のサンプルがあります。各サンプルには 4 つの特徴量と、3 つの可能なラベル名のいずれかがあります。
2. 後続の行はデータレコードであり、1 行につき 1 つの*[サンプル（example）](https://developers.google.com/machine-learning/glossary/#example)*が含まれます。
  * 最初の 4 つのフィールドは*[特徴量（features）](https://developers.google.com/machine-learning/glossary/#feature)*です。これらはサンプルの特性です。ここでは、花の測定値を表す浮動小数点数が保持されます。
  * 最後の列は*[ラベル（label）](https://developers.google.com/machine-learning/glossary/#label)*です。これは予測したい値です。このデータセットでは、花の名前に対応する 0、1、または 2 の整数値です。

これをコードで記述してみましょう。

In [ ]:
# CSV ファイル内の列の順序
column_names = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
    "species",
]

feature_names = column_names[:-1]
label_name = column_names[-1]

print(f"Features: {feature_names}")
print(f"Label: {label_name}")

各ラベルには文字列名（例えば "setosa"）が関連付けられていますが、機械学習は通常数値に依存します。ラベル番号は次のような名前付き表現にマッピングされます。

* `0`: Iris setosa
* `1`: Iris versicolor
* `2`: Iris virginica

特徴量とラベルの詳細については、[Machine Learning Crash Course の ML 用語セクション](https://developers.google.com/machine-learning/crash-course/framing/ml-terminology) を参照してください。

In [ ]:
class_names = ["Iris setosa", "Iris versicolor", "Iris virginica"]

### `tf.data.Dataset` の作成

TensorFlow の [Dataset API](../../guide/data.ipynb) は、モデルにデータを読み込むための一般的なケースの多くを処理します。これは、データを読み取り、訓練に使用する形式に変換するための高レベル API です。


データセットは CSV 形式のテキストファイルであるため、`tf.data.experimental.make_csv_dataset` 関数を使用してデータを適切な形式に解析します。この関数はモデルを訓練するためのデータを生成するため、デフォルトの動作ではデータをシャッフルし（`shuffle=True, shuffle_buffer_size=10000`）、データセットを無期限に繰り返します（`num_epochs=None`）。また、[batch_size](https://developers.google.com/machine-learning/glossary/#batch_size) パラメータも設定します。

In [ ]:
batch_size = 32

train_dataset = tf.data.experimental.make_csv_dataset(
    train_dataset_fp,
    batch_size,
    column_names=column_names,
    label_name=label_name,
    num_epochs=1,
)

`make_csv_dataset` 関数は `(features, label)` ペアの `tf.data.Dataset` を返します。ここで `features` は辞書 `{'feature_name': value}` です。

これらの `Dataset` オブジェクトは反復可能です。特徴量のバッチを見てみましょう。

In [ ]:
features, labels = next(iter(train_dataset))

print(features)

類似の特徴量がグループ化されている（*バッチ化されている*）ことに注目してください。各サンプル行のフィールドは、対応する特徴量配列に追加されます。`batch_size` を変更して、これらの特徴量配列に格納されるサンプルの数を設定します。

バッチからいくつかの特徴量をプロットすることで、いくつかのクラスタが見え始めます。

In [ ]:
plt.scatter(
    features["petal_length"],
    features["sepal_length"],
    c=labels,
    cmap="viridis",
)

plt.xlabel("Petal length")
plt.ylabel("Sepal length")
plt.show()

モデル構築ステップを簡素化するために、特徴量辞書を形状 `(batch_size, num_features)` の単一の配列に再パッケージ化する関数を作成します。

この関数は `tf.stack` メソッドを使用します。このメソッドはテンソルのリストから値を取得し、指定された次元で結合されたテンソルを作成します。

In [ ]:
def pack_features_vector(features, labels):
    """特徴量を単一の配列にパックします。"""
    features = tf.stack(list(features.values()), axis=1)
    return features, labels

次に、`tf.data.Dataset#map` メソッドを使用して、各 `(features,label)` ペアの `features` を訓練データセットにパックします。

In [ ]:
train_dataset = train_dataset.map(pack_features_vector)

`Dataset` の features 要素は、形状 `(batch_size, num_features)` の配列になりました。最初の数個のサンプルを見てみましょう。

In [ ]:
features, labels = next(iter(train_dataset))

print(features[:5])

## モデルタイプの選択

### なぜモデルが必要なのか？

*[モデル（model）](https://developers.google.com/machine-learning/crash-course/glossary#model)* とは、特徴量とラベルの間の関係性のことです。アヤメの分類問題の場合、モデルは萼片と花弁の測定値と予測されるアヤメの品種との間の関係を定義します。いくつかの単純なモデルは数行の代数式で記述できますが、複雑な機械学習モデルには要約が困難な多数のパラメータがあります。

機械学習を*使わずに*、4 つの特徴量とアヤメの品種との関係を特定できるでしょうか？つまり、従来のプログラミング手法（例えば、大量の条件分岐ステートメント）を使用してモデルを作成できるでしょうか？花弁と萼片の測定値と特定の品種との関係を特定できるほど十分に長い時間をかけてデータセットを分析すれば、可能かもしれません。しかし、より複雑なデータセットではこれは困難になり、場合によっては不可能になります。優れた機械学習のアプローチは、*ユーザーに代わってモデルを特定*してくれます。適切な機械学習モデルタイプに十分な代表的サンプルを入力すれば、プログラムが関係性を導き出してくれます。

### モデルの選択

訓練するモデルの種類を選択する必要があります。モデルには多くの種類があり、優れたモデルを選択するには経験が必要です。このチュートリアルでは、アヤメの分類問題を解決するためにニューラルネットワークを使用します。*[ニューラルネットワーク（neural network）](https://developers.google.com/machine-learning/glossary/#neural_network)* は、特徴量とラベルの間の複雑な関係を見つけることができます。これは、1 つ以上の*[隠れ層（hidden layers）](https://developers.google.com/machine-learning/glossary/#hidden_layer)*で構成された高度に構造化されたグラフです。各隠れ層は、1 つ以上の*[ニューロン（neurons）](https://developers.google.com/machine-learning/glossary/#neuron)*で構成されます。ニューラルネットワークにはいくつかのカテゴリがあり、このプログラムでは全結合（Dense）、すなわち*[全結合ニューラルネットワーク（fully-connected neural network）](https://developers.google.com/machine-learning/glossary/#fully_connected_layer)*を使用します。ある層のニューロンは、前の層の*すべての*ニューロンからの入力接続を受け取ります。例えば、図 2 は、入力層、2 つの隠れ層、および出力層で構成される全結合ニューラルネットワークを示しています。

<table>
  <tr><td>
    <img src="https://www.tensorflow.org/images/custom_estimators/full_network.png"
         alt="ネットワークアーキテクチャの図: 入力、2つの隠れ層、および出力">
  </td></tr>
  <tr><td align="center">
    <b>図 2.</b> 特徴量、隠れ層、予測を持つニューラルネットワーク。<br/>&nbsp;
  </td></tr>
</table>

図 2 のモデルが訓練され、ラベルのないサンプルが入力されると、3 つの予測（この花が特定のアヤメの品種である確率）が出力されます。この予測は*[推論（inference）](https://developers.google.com/machine-learning/crash-course/glossary#inference)*と呼ばれます。この例では、出力予測の合計は 1.0 になります。図 2 では、この予測の内訳は、*Iris setosa* が `0.02`、*Iris versicolor* が `0.95`、*Iris virginica* が `0.03` です。これは、モデルが 95% の確率で、ラベルのないサンプルの花が *Iris versicolor* であると予測していることを意味します。

### Keras を使用したモデルの作成

TensorFlow の `tf.keras` API は、モデルやレイヤを作成するための推奨される方法です。Keras がすべてを接続する複雑さを処理するため、モデルの構築や実験が容易になります。

`tf.keras.Sequential` モデルは、レイヤの線形スタックです。そのコンストラクタはレイヤインスタンスのリストを受け取ります。この場合、それぞれ 10 ノードの 2 つの `tf.keras.layers.Dense` レイヤと、ラベル予測を表す 3 ノードの出力レイヤです。最初のレイヤの `input_shape` パラメータはデータセットの特徴量の数に対応しており、必須です。

In [ ]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Dense(
            10, activation=tf.nn.relu, input_shape=(4,)
        ),  # 入力形状が必要
        tf.keras.layers.Dense(10, activation=tf.nn.relu),
        tf.keras.layers.Dense(3),
    ]
)

*[活性化関数（activation function）](https://developers.google.com/machine-learning/crash-course/glossary#activation_function)* は、レイヤ内の各ノードの出力形状を決定します。これらの非線形性は重要です。これらがないと、モデルは単一のレイヤと同等になってしまいます。多くの `tf.keras.activations` がありますが、隠れ層には [ReLU](https://developers.google.com/machine-learning/crash-course/glossary#ReLU) が一般的です。

隠れ層とニューロンの理想的な数は、問題とデータセットによって異なります。機械学習の多くの側面と同様に、ニューラルネットワークの最適な形状を選択するには知識と実験の組み合わせが必要です。経験則として、隠れ層とニューロンの数を増やすと通常はより強力なモデルが作成されますが、効果的に訓練するためにより多くのデータが必要になります。

### モデルの使用

このモデルが特徴量のバッチに対して何を行うかを簡単に見てみましょう。

In [ ]:
predictions = model(features)
predictions[:5]

ここで、各サンプルは各クラスの[ロジット（logit）](https://developers.google.com/machine-learning/crash-course/glossary#logits)を返します。

これらのロジットを各クラスの確率に変換するには、[softmax](https://developers.google.com/machine-learning/crash-course/glossary#softmax) 関数を使用します。

In [ ]:
tf.nn.softmax(predictions[:5])

クラス全体にわたって `tf.argmax` を取ると、予測されたクラスインデックスが得られます。ただし、モデルはまだ訓練されていないため、これらは良い予測ではありません。

In [ ]:
print(f"Prediction: {tf.argmax(predictions, axis=1)}")
print(f"    Labels: {labels}")

## モデルの訓練

*[訓練（training）](https://developers.google.com/machine-learning/crash-course/glossary#training)* とは、モデルが徐々に最適化される、またはモデルがデータセットを*学習*する機械学習の段階です。目標は、訓練データセットの構造について十分に学習し、未知のデータについて予測を行えるようにすることです。訓練データセットについて*過剰に*学習してしまうと、予測は学習したデータに対してのみ機能し、汎化できなくなります。この問題は*[過学習（overfitting）](https://developers.google.com/machine-learning/crash-course/glossary#overfitting)*と呼ばれます。これは、問題の解決方法を理解するのではなく、答えを丸暗記するようなものです。

アヤメの分類問題は、*[教師あり機械学習（supervised machine learning）](https://developers.google.com/machine-learning/glossary/#supervised_machine_learning)*の例です。モデルはラベルを含むサンプルから訓練されます。*[教師なし機械学習（unsupervised machine learning）](https://developers.google.com/machine-learning/glossary/#unsupervised_machine_learning)*では、サンプルにラベルが含まれません。代わりに、モデルは通常特徴量間のパターンを見つけ出します。

### 損失関数と勾配関数の定義

訓練と評価の両方の段階で、モデルの*[損失（loss）](https://developers.google.com/machine-learning/crash-course/glossary#loss)*を計算する必要があります。これは、モデルの予測が目的のラベルからどれだけ離れているか、言い換えればモデルのパフォーマンスがどれほど悪いかを測定します。この値を最小化（最適化）することが目標です。

このモデルでは、モデルのクラス確率予測と目的のラベルを受け取り、サンプル全体の平均損失を返す `tf.keras.losses.SparseCategoricalCrossentropy` 関数を使用して損失を計算します。

In [ ]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

In [ ]:
def loss(model, x, y, training):
    # training=training は、訓練時と推論時で動作が異なるレイヤ
    # （例: Dropout）がある場合にのみ必要です。
    y_ = model(x, training=training)

    return loss_object(y_true=y, y_pred=y_)


l = loss(model, features, labels, training=False)
print(f"Loss test: {l}")

モデルの最適化に使用される*[勾配（gradients）](https://developers.google.com/machine-learning/crash-course/glossary#gradient)*を計算するには、`tf.GradientTape` コンテキストを使用します。

In [ ]:
def grad(model, inputs, targets):
    with tf.GradientTape() as tape:
        loss_value = loss(model, inputs, targets, training=True)
    return loss_value, tape.gradient(loss_value, model.trainable_variables)

### オプティマイザの作成

*[オプティマイザ（optimizer）](https://developers.google.com/machine-learning/crash-course/glossary#optimizer)*は、計算された勾配をモデルの変数に適用して `loss` 関数を最小化します。損失関数は曲面（図 3 を参照）と考えることができ、歩き回りながらその最も低い点を見つけたいと考えます。勾配は最も急な上り坂の方向を指しているため、逆方向に進んで坂を下ります。各バッチの損失と勾配を繰り返し計算することで、訓練中にモデルを調整します。徐々に、モデルは損失を最小化する重みとバイアスの最適な組み合わせを見つけます。そして損失が低くなるほど、モデルの予測精度は向上します。

<table>
  <tr><td>
    <img src="https://cs231n.github.io/assets/nn3/opt1.gif" width="70%"
         alt="3次元空間で時間の経過とともに可視化された最適化アルゴリズム。">
  </td></tr>
  <tr><td align="center">
    <b>図 3.</b> 3 次元空間で時間の経過とともに可視化された最適化アルゴリズム。<br/>（出典: <a href="http://cs231n.github.io/neural-networks-3/">Stanford class CS231n</a>、MIT ライセンス、画像提供: <a href="https://twitter.com/alecrad">Alec Radford</a>）
  </td></tr>
</table>

TensorFlow には、訓練に使用できる多くの最適化アルゴリズムが用意されています。このモデルでは、*[確率的勾配降下法](https://developers.google.com/machine-learning/crash-course/glossary#gradient_descent)*（SGD）アルゴリズムを実装する `tf.keras.optimizers.SGD` を使用します。`learning_rate` は、坂を下る各イテレーションで進むステップサイズを設定します。これは、より良い結果を得るために通常調整する*ハイパーパラメータ*です。

オプティマイザを設定しましょう。

In [ ]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.01)

これを使用して、単一の最適化ステップを計算します。

In [ ]:
loss_value, grads = grad(model, features, labels)

print(
    "Step: {}, Initial Loss: {}".format(
        optimizer.iterations.numpy(), loss_value.numpy()
    )
)

optimizer.apply_gradients(zip(grads, model.trainable_variables))

print(
    "Step: {},         Loss: {}".format(
        optimizer.iterations.numpy(),
        loss(model, features, labels, training=True).numpy(),
    )
)

### 訓練ループ

すべてのパーツが揃い、モデルの訓練の準備が整いました。訓練ループは、データセットのサンプルをモデルに供給して、より良い予測ができるように支援します。次のコードブロックは、以下の訓練ステップを設定します。

1. 各*エポック*を反復処理します。エポックとは、データセットを 1 回通過することです。
2. エポック内で、訓練 `Dataset` 内の各サンプルを反復処理し、その*特徴量*（`x`）と*ラベル*（`y`）を取得します。
3. サンプルの特徴量を使用して予測を行い、ラベルと比較します。予測の不正確さを測定し、それを使用してモデルの損失と勾配を計算します。
4. `optimizer` を使用してモデルの変数を更新します。
5. 可視化のためにいくつかの統計情報を追跡します。
6. 各エポックに対して繰り返します。

`num_epochs` 変数は、データセットコレクションをループする回数です。直感に反して、モデルを長く訓練しても、より良いモデルが保証されるわけではありません。`num_epochs` は、調整可能な*[ハイパーパラメータ（hyperparameter）](https://developers.google.com/machine-learning/glossary/#hyperparameter)*です。適切な数値を選択するには、通常経験と実験の両方が必要です。

In [ ]:
## 注: このセルを再実行すると同じモデル変数が使用されます

# プロット用に結果を保持
train_loss_results = []
train_accuracy_results = []

num_epochs = 201

for epoch in range(num_epochs):
    epoch_loss_avg = tf.keras.metrics.Mean()
    epoch_accuracy = tf.keras.metrics.SparseCategoricalAccuracy()

    # 訓練ループ - バッチサイズ 32 を使用
    for x, y in train_dataset:
        # モデルを最適化
        loss_value, grads = grad(model, x, y)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        # 進行状況を追跡
        epoch_loss_avg.update_state(loss_value)  # 現在のバッチ損失を追加
        # 予測ラベルと実際のラベルを比較
        # training=True は、訓練時と推論時で動作が異なるレイヤ
        # （例: Dropout）がある場合にのみ必要です。
        epoch_accuracy.update_state(y, model(x, training=True))

    # エポックの終了
    train_loss_results.append(epoch_loss_avg.result())
    train_accuracy_results.append(epoch_accuracy.result())

    if epoch % 50 == 0:
        print(
            "Epoch {:03d}: Loss: {:.3f}, Accuracy: {:.3%}".format(
                epoch, epoch_loss_avg.result(), epoch_accuracy.result()
            )
        )

### 損失関数の推移の可視化

モデルの訓練の進行状況を出力することも役立ちますが、この進行状況を確認できると*さらに*役立ちます。[TensorBoard](https://www.tensorflow.org/tensorboard) は TensorFlow にパッケージ化されている優れた可視化ツールですが、`matplotlib` モジュールを使用して基本的なグラフを作成することもできます。

これらのグラフを解釈するにはある程度の経験が必要ですが、基本的には*損失*が低下し、*精度*が上昇することを確認したいと考えます。

In [ ]:
fig, axes = plt.subplots(2, sharex=True, figsize=(12, 8))
fig.suptitle("Training Metrics")

axes[0].set_ylabel("Loss", fontsize=14)
axes[0].plot(train_loss_results)

axes[1].set_ylabel("Accuracy", fontsize=14)
axes[1].set_xlabel("Epoch", fontsize=14)
axes[1].plot(train_accuracy_results)
plt.show()

## モデルの有効性の評価

モデルが訓練されたので、そのパフォーマンスに関するいくつかの統計情報を取得できます。

*評価*とは、モデルがどれだけ効果的に予測を行うかを判断することを意味します。アヤメの分類におけるモデルの有効性を判断するには、いくつかの萼片と花弁の測定値をモデルに渡し、それらが表すアヤメの品種を予測するようにモデルに求めます。次に、モデルの予測と実際のラベルを比較します。例えば、入力サンプルの半分で正しい品種を選択したモデルの*[正解率（accuracy）](https://developers.google.com/machine-learning/glossary/#accuracy)*は `0.5` です。図 4 は、5 つの予測のうち 4 つを正解し、80% の正解率を達成した、やや効果的なモデルを示しています。

<table cellpadding="8" border="0">
  <colgroup>
    <col span="4" >
    <col span="1" bgcolor="lightblue">
    <col span="1" bgcolor="lightgreen">
  </colgroup>
  <tr bgcolor="lightgray">
    <th colspan="4">サンプルの特徴量</th>
    <th colspan="1">ラベル</th>
    <th colspan="1" >モデルの予測</th>
  </tr>
  <tr>
    <td>5.9</td><td>3.0</td><td>4.3</td><td>1.5</td><td align="center">1</td><td align="center">1</td>
  </tr>
  <tr>
    <td>6.9</td><td>3.1</td><td>5.4</td><td>2.1</td><td align="center">2</td><td align="center">2</td>
  </tr>
  <tr>
    <td>5.1</td><td>3.3</td><td>1.7</td><td>0.5</td><td align="center">0</td><td align="center">0</td>
  </tr>
  <tr>
    <td>6.0</td> <td>3.4</td> <td>4.5</td> <td>1.6</td> <td align="center">1</td><td align="center" bgcolor="red">2</td>
  </tr>
  <tr>
    <td>5.5</td><td>2.5</td><td>4.0</td><td>1.3</td><td align="center">1</td><td align="center">1</td>
  </tr>
  <tr><td align="center" colspan="6">
    <b>図 4.</b> 正解率 80% のアヤメ分類器。<br/>&nbsp;
  </td></tr>
</table>

### テストデータセットのセットアップ

モデルの評価は、モデルの訓練と似ています。最大の相違点は、サンプルが訓練セットではなく別の*[テストセット（test set）](https://developers.google.com/machine-learning/crash-course/glossary#test_set)*から提供される点です。モデルの有効性を公平に評価するには、モデルの評価に使用するサンプルが、モデルの訓練に使用されたサンプルと異なっている必要があります。

テスト用 `Dataset` のセットアップは、訓練用 `Dataset` のセットアップと似ています。CSV テキストファイルをダウンロードしてその値を解析し、少しシャッフルします。

In [ ]:
test_url = (
    "https://storage.googleapis.com/download.tensorflow.org/data/iris_test.csv"
)

test_fp = tf.keras.utils.get_file(
    fname=os.path.basename(test_url), origin=test_url
)

In [ ]:
test_dataset = tf.data.experimental.make_csv_dataset(
    test_fp,
    batch_size,
    column_names=column_names,
    label_name="species",
    num_epochs=1,
    shuffle=False,
)

test_dataset = test_dataset.map(pack_features_vector)

### テストデータセットでのモデルの評価

訓練段階とは異なり、モデルはテストデータの単一の [エポック（epoch）](https://developers.google.com/machine-learning/glossary/#epoch) のみを評価します。次のコードセルでは、テストセット内の各サンプルを反復処理し、モデルの予測と実際のラベルを比較します。これは、テストセット全体にわたるモデルの正解率を測定するために使用されます。

In [ ]:
test_accuracy = tf.keras.metrics.Accuracy()

for x, y in test_dataset:
    # training=False は、訓練時と推論時で動作が異なるレイヤ
    # （例: Dropout）がある場合にのみ必要です。
    logits = model(x, training=False)
    prediction = tf.argmax(logits, axis=1, output_type=tf.int32)
    test_accuracy(prediction, y)

print(f"Test set accuracy: {test_accuracy.result():.3%}")

最後のバッチを見ると、例えばモデルはおおむね正解していることがわかります。

In [ ]:
tf.stack([y, prediction], axis=1)

## 訓練済みモデルを使用した予測の実行

モデルを訓練し、アヤメの品種を分類するのに優れている（完璧ではありませんが）ことが「実証」されました。ここで、訓練済みモデルを使用して、[ラベルのないサンプル（unlabeled examples）](https://developers.google.com/machine-learning/glossary/#unlabeled_example)、つまり特徴量は含んでいるがラベルを含んでいないサンプルに対して予測を行ってみましょう。

実社会では、ラベルのないサンプルはアプリ、CSV ファイル、データフィードなど、さまざまなソースから提供される可能性があります。ここでは、手動で 3 つのラベルなしサンプルを提供して、そのラベルを予測します。ラベル番号は、次のような名前付き表現にマッピングされていることを思い出してください。

* `0`: Iris setosa
* `1`: Iris versicolor
* `2`: Iris virginica

In [ ]:
predict_dataset = tf.convert_to_tensor(
    [
        [5.1, 3.3, 1.7, 0.5],
        [5.9, 3.0, 4.2, 1.5],
        [6.9, 3.1, 5.4, 2.1],
    ]
)

# training=False は、訓練時と推論時で動作が異なるレイヤ
# （例: Dropout）がある場合にのみ必要です。
predictions = model(predict_dataset, training=False)

for i, logits in enumerate(predictions):
    class_idx = tf.argmax(logits).numpy()
    p = tf.nn.softmax(logits)[class_idx]
    name = class_names[class_idx]
    print(f"Example {i} prediction: {name} ({100 * p:4.1f}%)")